In [1]:
!git clone https://github.com/paulphilip-louis/LLM_NeuralExec.git

Cloning into 'LLM_NeuralExec'...
remote: Enumerating objects: 172, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 172 (delta 79), reused 118 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (172/172), 2.10 MiB | 14.53 MiB/s, done.
Resolving deltas: 100% (79/79), done.


In [2]:
!pwd

/content


In [5]:
import os
os.chdir("/content/LLM_NeuralExec")

In [6]:
import os, sys, importlib, argparse
import random

from NeuralExec.llm import load_llm
from NeuralExec.discrete_opt import WhiteBoxTokensOpt
from NeuralExec.utility import read_pickle, write_pickle
from NeuralExec.logger import Logger

def init_opt(wbo, conf_file, hparams):
    conf_file_name = conf_file.split('.')[-1]
    log_path = os.path.join(hparams['result_dir'], conf_file_name)

    if os.path.isfile(log_path):
        print(f"Resuming opt {log_path}")
        logger = read_pickle(log_path)
        hparams = logger.confs
        ne, _ = logger.get_last_adv_tok(best=True)
    else:
        print(f"Init opt {log_path}")
        # init/load log file
        logger = Logger(hparams)
        # init Neural Exec
        if 'boostrap_seed' in hparams:
            print("init_adv_seg boostrapping...")
            ne = wbo.init_adv_seg_boot(*hparams['boostrap_seed'], hparams['sep'])
        else:
            print("NexuralExec Random init...")
            ne = wbo.init_adv_seg(hparams['prefix_size'], hparams['postfix_size'], hparams['sep'])
    return ne, logger, log_path, hparams

def sample_batch(training_prompts, batch_size):
    return random.choices(training_prompts, k=batch_size)


/content/LLM_NeuralExec/NeuralExec/utility.py:34: SyntaxWarning: invalid escape sequence '\d'
  alpha = re.compile('^[A-Za-z\d.,;:?!-]+$')


In [8]:
# set gpus
os.environ['CUDA_VISIBLE_DEVICES'] = "0"

# load conf file
conf = importlib.import_module("confs.15_5_qwen25_15b")
hparams = conf.hparams

# load data
train_path, eval_path = hparams['dataset_paths']
training_prompts, validation_prompts = read_pickle(train_path), read_pickle(eval_path)

# load LLM
print(f"Loading {hparams['llm']}...")
llm = load_llm(hparams['llm'])

# setup opt class
wbo = WhiteBoxTokensOpt(llm, hparams)

# init opt
ne, logger, log_path, hparams = init_opt(wbo, "confs.15_5_qwen25_15b", hparams)
wbo.hparams = hparams

# opt loop
for i in range(hparams['number_of_rounds']):
    print(f'Start round {i+1}/{hparams["number_of_rounds"]}')

    if i % hparams['eval_fq'] == 0:
        print("Starting evaluation...")
        eval_losses = wbo.eval_loss(validation_prompts, ne)
        print(eval_losses)
        logger.add_eval_log(ne, eval_losses, wbo.tokenizer)
        print("end evaluation.")

        logger.candidate_pool.insert_candidate(ne, eval_losses.mean())
        ne, best_loss_pool = logger.candidate_pool.get_best()
        print(eval_losses.mean(), best_loss_pool)

        write_pickle(log_path, logger)


    # sample batch for gradient
    train_batch = sample_batch(training_prompts, hparams['gradient_batch_size'])
    # compute gradient
    print("Computing gradient...")
    gradient, loss, losses = wbo.get_gradient_accum(ne, train_batch)
    logger.add_train_log(loss, ne, wbo.tokenizer)

    # sample candidate solutions
    new_candidate_tok = wbo.sample_new_candidates(ne, gradient)
    # filter out bad ones
    new_candidate_tok = wbo.filter_candidates(ne, new_candidate_tok)
    # pick new solution
    ne, best_candidate_loss, _, _ = wbo.test_candidates(train_batch, new_candidate_tok)


Loading Qwen/Qwen2.5-1.5B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

NotImplementedError: Could not run 'aten::empty.memory_format' with arguments from the 'MPS' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'aten::empty.memory_format' is only available for these backends: [CPU, Meta, QuantizedCPU, QuantizedMeta, MkldnnCPU, SparseCPU, SparseMeta, SparseCsrCPU, SparseCsrMeta, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Named, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradOther, AutogradCPU, AutogradCUDA, AutogradHIP, AutogradXLA, AutogradMPS, AutogradIPU, AutogradXPU, AutogradHPU, AutogradVE, AutogradLazy, AutogradMTIA, AutogradMAIA, AutogradPrivateUse1, AutogradPrivateUse2, AutogradPrivateUse3, AutogradMeta, AutogradNestedTensor, Tracer, AutocastCPU, AutocastMTIA, AutocastMAIA, AutocastX